# webgpu-q — real-GPU WebGPU probe on a free Colab T4

Confirms a **non-fallback** WebGPU adapter (rejects SwiftShader / llvmpipe /
lavapipe) on a real NVIDIA **T4**, runs webgpu-q's **L1 + L3** GPU kernels
against the live deployed site, reports each protocol's pass/fail, and **saves +
downloads the full artifact JSON**. Background:
[`docs/webgpu-ci-providers.md`](https://github.com/abgnydn/webgpu-q/blob/claude/blissful-davinci-7zm63h/docs/webgpu-ci-providers.md).

### Do this first
1. **Runtime → Change runtime type → T4 GPU → Save**
2. **Runtime → Run all**

The gate is strict on purpose: if it lands on software it prints **❌** and
**skips the kernels** rather than emitting fake "GPU" numbers.

In [ ]:
#@title 1) Setup — Chromium + Vulkan + NVIDIA userspace  (~2 min, run once)
import subprocess

smi = subprocess.run(
    ["bash", "-lc", "nvidia-smi --query-gpu=name,driver_version --format=csv,noheader"],
    capture_output=True, text=True).stdout.strip()
print("GPU:", smi or "*** NONE — set Runtime → Change runtime type → T4 GPU ***")
drv = (smi.split(",")[-1].strip().split(".")[0]) if smi else "535"

# vulkan loader + tools, dbus (Chrome needs it), and the NVIDIA GL/Vulkan
# *userspace* (the load-bearing bit — compute driver alone => llvmpipe).
get_ipython().system("apt-get update -qq")
get_ipython().system(f"apt-get install -y -qq vulkan-tools libvulkan1 dbus libnvidia-gl-{drv} "
                     f"|| apt-get install -y -qq vulkan-tools libvulkan1 dbus")
get_ipython().system("pip -q install playwright")
get_ipython().system("playwright install --with-deps chromium")  # --with-deps: apt-install libatk & co
get_ipython().system("service dbus start 2>/dev/null || true")
print("\\n--- vulkaninfo (must show NVIDIA, NOT llvmpipe/lavapipe) ---")
get_ipython().system("vulkaninfo 2>/dev/null | grep -iE 'deviceName|driverName' | head || echo '(no vulkaninfo)'")

In [ ]:
#@title 2) Gate → run L1/L3 → report + save/download artifacts  (async, Colab-safe)
import json
from playwright.async_api import async_playwright   # async: Colab already runs an asyncio loop

URL = "https://webgpu-q.vercel.app/experiments/?peak=320"  # T4 GDDR6 peak BW; Chrome masks the device name
LEVELS = ["1", "3"]   # add "E34" for the (T) CPU/GPU bench (heavy, minutes)

CHROME_ARGS = [
    "--no-sandbox", "--headless=new", "--use-angle=vulkan", "--enable-features=Vulkan",
    "--disable-vulkan-surface", "--enable-unsafe-webgpu",
    "--enable-dawn-features=allow_unsafe_apis,disable_adapter_blocklist",
]
GATE_JS = r"""
async () => {
  if (!('gpu' in navigator) || !navigator.gpu) return { ok:false, reason:'no navigator.gpu' };
  let a = null;
  try { a = await navigator.gpu.requestAdapter({ powerPreference:'high-performance' }); } catch (e) {}
  if (!a) { try { a = await navigator.gpu.requestAdapter(); } catch (e) {} }
  if (!a) return { ok:false, reason:'requestAdapter returned null' };
  const info = a.info ? a.info : (a.requestAdapterInfo ? await a.requestAdapterInfo() : {});
  const fb = (a.info && 'isFallbackAdapter' in a.info) ? a.info.isFallbackAdapter
           : ('isFallbackAdapter' in a) ? a.isFallbackAdapter : null;
  return { ok:true, vendor:info.vendor||'', architecture:info.architecture||'',
           device:info.device||'', description:info.description||'', isFallbackAdapter:fb };
}
"""
SOFTWARE = ("swiftshader", "llvmpipe", "lavapipe", "software", "google inc.")

def report(name, art):
    print(f"\\n########## {name}: status={art.get('status')} ##########")
    if art.get("diagnosis"):
        print("diagnosis:", art["diagnosis"])
    subs = art.get("artifacts") if isinstance(art.get("artifacts"), list) else [art]
    for a in subs:
        rows = a.get("rows", [])
        fails = [r for r in rows if r.get("passed") is False]
        fs = [r.get("fidelityMin", r.get("fidelityMedian")) for r in rows]
        fs = [x for x in fs if isinstance(x, (int, float))]
        proto = a.get("meta", {}).get("protocol", "?")
        print(f"  [{proto}] status={a.get('status')} rows={len(rows)} FAILS={len(fails)}"
              + (f" worstF={min(fs):.9f}" if fs else "")
              + ((" | " + a["diagnosis"]) if a.get("diagnosis") else ""))
        for r in fails[:12]:
            print("    FAIL:", json.dumps({k: r.get(k) for k in
                  ('circuit', 'nQubits', 'depth', 'fidelityMedian', 'fidelityMin', 'normGpuMedian', 'notes')}))

async def run():
    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=False,  # we pass --headless=new ourselves
            args=CHROME_ARGS,
            # strip Playwright's injected software-fallback args so the gate can't pass on SwiftShader
            ignore_default_args=["--use-angle=swiftshader-webgl", "--enable-unsafe-swiftshader"],
        )
        page = await browser.new_page()
        page.set_default_timeout(0)
        await page.goto(URL, wait_until="domcontentloaded", timeout=120_000)

        info = await page.evaluate(GATE_JS)
        blob = " ".join(str(info.get(k, "")) for k in ("vendor", "architecture", "device", "description")).lower()
        real = bool(info.get("ok")) and info.get("isFallbackAdapter") is not True and not any(s in blob for s in SOFTWARE)
        print(json.dumps(info, indent=2))
        print("\\n" + ("✅ REAL GPU" if real else "❌ SOFTWARE / none — rejected (gate working)"),
              "— vendor=", repr(info.get("vendor")), "desc=", repr(info.get("description")))

        saved = []
        if real and LEVELS:
            await page.wait_for_function("window.__webgpuq && window.__webgpuq.ready === true", timeout=120_000)
            for lv in LEVELS:
                fn = "runE34" if lv.upper() == "E34" else f"runLevel{lv}"
                art = await page.evaluate(f"async () => await window.__webgpuq.{fn}()")
                path = f"/content/webgpu-q-T4-{fn}.json"
                with open(path, "w") as f:
                    json.dump(art, f)
                saved.append(path)
                report(fn, art)
        await browser.close()

    for path in saved:
        try:
            from google.colab import files
            files.download(path)
        except Exception as e:
            print(f"(auto-download skipped for {path}: {e} — it's in /content/)")
    return None

await run()